<a href="https://colab.research.google.com/github/Rstam59/TaskDataRepoForStudents/blob/main/ASR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tarfile

with tarfile.open("/content/1774118365245-cv-corpus-25.0-2026-03-09-az.tar.gz", "r:gz") as tar:
    tar.extractall("common_voice_az")

In [ ]:
import os
import pandas as pd
from datasets import Dataset, Audio

DATA_DIR = "/content/common_voice_az/cv-corpus-25.0-2026-03-09/az"
CLIPS_DIR = f"{DATA_DIR}/clips"

df = pd.read_csv(f"{DATA_DIR}/train.tsv", sep="\t")

# choose 50 test samples
df = df.sample(50, random_state=42).copy()

# make full audio paths
df["audio"] = df["path"].apply(lambda x: os.path.join(CLIPS_DIR, x))

# keep only columns we need
df = df[["audio", "sentence"]]

dataset = Dataset.from_pandas(df)

# Hugging Face will decode/resample audio for you
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

dataset[0]

In [ ]:
sample = dataset[0]

audio_array = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
text = sample["sentence"]

print(audio_array.shape)
print(sampling_rate)
print(text)

In [ ]:
from IPython.display import Audio as IPythonAudio

IPythonAudio(audio_array, rate=sampling_rate)

In [ ]:
!pip install -q gradio librosa

In [ ]:
import gradio as gr

def generate_audio():
    example = dataset.shuffle(seed=None)[0]
    audio = example["audio"]

    return (
        audio["sampling_rate"],
        audio["array"],
    ), example["sentence"]


with gr.Blocks() as demo:
    gr.Markdown("## Common Voice Azerbaijani — Random Test Samples")

    with gr.Column():
        for _ in range(4):
            audio, sentence = generate_audio()
            gr.Audio(value=audio, label=sentence)

demo.launch(debug=True)

In [ ]:
import librosa
import matplotlib.pyplot as plt

example = dataset[6]

array = example["audio"]["array"]
sampling_rate = example["audio"]["sampling_rate"]

print("Transcript:", example["sentence"])
print("Sampling rate:", sampling_rate)
print("Audio shape:", array.shape)

plt.figure(figsize=(12, 4))
librosa.display.waveshow(array, sr=sampling_rate)
plt.title("Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


D = librosa.stft(array)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

plt.figure(figsize=(12, 4))
librosa.display.specshow(
    S_db,
    sr=sampling_rate,
    x_axis="time",
    y_axis="hz"
)
plt.colorbar(format="%+2.0f dB")
plt.title("Spectrogram")
plt.show()

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

example = dataset[0]

array = example["audio"]["array"]
sampling_rate = example["audio"]["sampling_rate"]

mel = librosa.feature.melspectrogram(
    y=array,
    sr=sampling_rate,
    n_mels=80,
    fmax=8000
)

mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(12, 4))
librosa.display.specshow(
    mel_db,
    sr=sampling_rate,
    x_axis="time",
    y_axis="mel",
    fmax=8000
)
plt.colorbar(format="%+2.0f dB")
plt.title("Mel Spectrogram")
plt.show()

In [ ]:
import random
from IPython.display import Audio as IPythonAudio, display

idx = random.randint(0, len(dataset) - 1)
example = dataset[idx]

array = example["audio"]["array"]
sampling_rate = example["audio"]["sampling_rate"]

print("Index:", idx)
print("Transcript:", example["sentence"])
display(IPythonAudio(array, rate=sampling_rate))

plt.figure(figsize=(12, 4))
librosa.display.waveshow(array, sr=sampling_rate)
plt.title(f"Waveform — sample {idx}")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
!pip install -q transformers accelerate datasets evaluate jiwer librosa soundfile

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

model.eval()

print("Device:", device)

In [ ]:
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="azerbaijani",
    task="transcribe"
)

In [ ]:
predictions = []
references = []

for i, example in enumerate(dataset):
    audio = example["audio"]

    input_features = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids
        )

    pred_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    ref_text = example["sentence"]

    predictions.append(pred_text)
    references.append(ref_text)

    print(f"{i+1}/{len(dataset)}")
    print("REF :", ref_text)
    print("PRED:", pred_text)
    print("-" * 80)

In [ ]:
import re

def normalize_text(text):
    text = text.lower().strip()

    # keep Azerbaijani letters, numbers, and spaces
    text = re.sub(r"[^\w\səıöüçğş]", " ", text)

    # normalize multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


norm_predictions = [normalize_text(x) for x in predictions]
norm_references = [normalize_text(x) for x in references]

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

avg_wer = wer_metric.compute(
    predictions=norm_predictions,
    references=norm_references
)

avg_cer = cer_metric.compute(
    predictions=norm_predictions,
    references=norm_references
)

print(f"Average WER: {avg_wer * 100:.2f}%")
print(f"Average CER: {avg_cer * 100:.2f}%")

In [ ]:
from jiwer import wer, cer
import pandas as pd

rows = []

for i, (ref, pred, norm_ref, norm_pred) in enumerate(
    zip(references, predictions, norm_references, norm_predictions)
):
    sample_wer = wer(norm_ref, norm_pred)
    sample_cer = cer(norm_ref, norm_pred)

    rows.append({
        "index": i,
        "reference": ref,
        "prediction": pred,
        "normalized_reference": norm_ref,
        "normalized_prediction": norm_pred,
        "wer_percent": sample_wer * 100,
        "cer_percent": sample_cer * 100,
    })

results_df = pd.DataFrame(rows)

results_df.head()

In [ ]:
best_5 = results_df.sort_values("wer_percent", ascending=True).head(5)

best_5[[
    "index",
    "wer_percent",
    "cer_percent",
    "reference",
    "prediction"
]]

In [ ]:
worst_5 = results_df.sort_values("wer_percent", ascending=False).head(5)

worst_5[[
    "index",
    "wer_percent",
    "cer_percent",
    "reference",
    "prediction"
]]

In [ ]:
print("Model: openai/whisper-small")
print(f"Number of samples: {len(results_df)}")
print(f"Average WER (%): {avg_wer * 100:.2f}")
print(f"Average CER (%): {avg_cer * 100:.2f}")

print("\nBest 5 examples:")
display(best_5[["index", "wer_percent", "cer_percent", "reference", "prediction"]])

print("\nWorst 5 examples:")
display(worst_5[["index", "wer_percent", "cer_percent", "reference", "prediction"]])

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/test.tsv", sep="\t")


df = df.copy()


df["audio"] = df["path"].apply(lambda x: os.path.join(CLIPS_DIR, x))

df = df[["audio", "sentence"]]

dataset = Dataset.from_pandas(df)


dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

full_dataset = dataset.shuffle(seed=42)

train_size = min(200, int(len(full_dataset) * 0.8))
eval_size = min(50, len(full_dataset) - train_size)

train_dataset = full_dataset.select(range(train_size))
eval_dataset = full_dataset.select(range(train_size, train_size + eval_size))

print(train_dataset)
print(eval_dataset)

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="azerbaijani",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(device)

model.generation_config.language = "azerbaijani"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="azerbaijani",
    task="transcribe"
)

model.generation_config.suppress_tokens = []

In [ ]:
def prepare_example(example):
    audio = example["audio"]

    example["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    example["labels"] = processor.tokenizer(
        example["sentence"]
    ).input_ids

    return example


train_dataset = train_dataset.map(
    prepare_example,
    remove_columns=train_dataset.column_names
)

eval_dataset = eval_dataset.map(
    prepare_example,
    remove_columns=eval_dataset.column_names
)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [
            {"input_ids": f["labels"]} for f in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorSpeechSeq2Seq(processor=processor)

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_texts = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_texts = processor.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    pred_texts = [normalize_text(x) for x in pred_texts]
    label_texts = [normalize_text(x) for x in label_texts]

    wer = wer_metric.compute(
        predictions=pred_texts,
        references=label_texts
    )

    return {"wer": wer}

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-small-az-finetuned",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=1e-5,
    warmup_steps=10,

    max_steps=100,

    eval_strategy="steps",
    eval_steps=20,

    save_strategy="steps",
    save_steps=20,

    logging_steps=10,

    predict_with_generate=True,
    generation_max_length=128,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
trainer.save_model("/content/whisper-small-az-finetuned")
processor.save_pretrained("/content/whisper-small-az-finetuned")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

logs = pd.DataFrame(trainer.state.log_history)
logs.head()

In [ ]:
train_logs = logs[logs["loss"].notna()] if "loss" in logs else pd.DataFrame()
eval_logs = logs[logs["eval_loss"].notna()] if "eval_loss" in logs else pd.DataFrame()

plt.figure(figsize=(8, 4))

if len(train_logs) > 0:
    plt.plot(train_logs["step"], train_logs["loss"], label="train loss")

if len(eval_logs) > 0:
    plt.plot(eval_logs["step"], eval_logs["eval_loss"], label="validation loss")

plt.xlabel("step")
plt.ylabel("loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

In [ ]:
wer_logs = logs[logs["eval_wer"].notna()] if "eval_wer" in logs else pd.DataFrame()

plt.figure(figsize=(8, 4))
plt.plot(wer_logs["step"], wer_logs["eval_wer"] * 100, marker="o")
plt.xlabel("step")
plt.ylabel("WER (%)")
plt.title("Validation WER during fine-tuning")
plt.show()